# Simplest GPT2-style Small Transformer on BabiStories
Decoder-only causal language model. Compact and based on the previous notebook style.

In [13]:
import torch,random,math,re,json
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cfg={
    "d_model":128,
    "d_ff":512,
    "num_heads":4,
    "num_layers":4,
    "max_vocab":12000,
    "seq_len":128,
    "batch_size":4,
    "lr":3e-4,
    "epochs":100,
    "dropout":0.05,
    "weight_decay":0.1,
    "use_wq":False,
    "use_aq":False,
    "w_bits":8,
    "a_bits":8,
    "use_ws":False,
    "ws_ratio":0.0,
    "attn_top_k":None,
    "norm":"pre"
}

cuda


## Data

#### Load BabiStories Dataset

In [14]:
def read_texts_from_file(p):
    texts=[]
    if p.suffix.lower()==".txt":
        x=p.read_text(encoding="utf-8",errors="ignore")
        for t in re.split(r"\n\s*\n|\n",x):
            t=" ".join(t.split())
            if len(t.split())>20:texts.append(t)
    elif p.suffix.lower()==".jsonl":
        for line in p.open(encoding="utf-8",errors="ignore"):
            if line.strip():
                d=json.loads(line);t=d.get("text") or d.get("story") or d.get("content") or ""
                t=" ".join(t.split())
                if len(t.split())>20:texts.append(t)
    elif p.suffix.lower()==".json":
        d=json.loads(p.read_text(encoding="utf-8",errors="ignore"))
        if isinstance(d,list):
            for e in d:
                t=e.get("text") or e.get("story") or e.get("content") or ""
                t=" ".join(t.split())
                if len(t.split())>20:texts.append(t)
    return texts

def load_babistories(folder="BabiStories/data/extracted"):
    folder=Path(folder);texts=[]
    if not folder.exists():raise FileNotFoundError(folder)
    for p in folder.rglob("*"):
        if p.suffix.lower() in [".txt",".jsonl",".json"]:texts+=read_texts_from_file(p)
    if len(texts)==0:raise ValueError("No .txt/.json/.jsonl stories found")
    return texts

texts=load_babistories()
random.seed(42);random.shuffle(texts)
print("texts:",len(texts))
print(texts[0][:300])

texts: 2221391
"In a hidden corner of the playground, a family of books sat quietly. Quinci, a bright red book with big white letters, was the smallest of the bunch. She always felt miserable because she was often overlooked due to her size. One sunny afternoon, a young girl named Mia came to the playground. She w


In [15]:
def build_context(texts,cfg):
    sp=["<PAD>","<EOS>","<UNK>"];c=Counter()
    for t in texts:c.update(t.split())
    vocab=sp+[w for w,_ in c.most_common(cfg["max_vocab"]-len(sp))]
    stoi={w:i for i,w in enumerate(vocab)};itos={i:w for w,i in stoi.items()}
    return {"stoi":stoi,"itos":itos,"vocab":vocab,"pad":stoi["<PAD>"],"eos":stoi["<EOS>"],"unk":stoi["<UNK>"],"seq_len":cfg["seq_len"]}

def make_lm_examples(texts,ctx,max_examples=20000,stride=64):
    ex=[]
    for t in texts:
        ids=[ctx["stoi"].get(w,ctx["unk"]) for w in t.split()]+[ctx["eos"]]
        if len(ids)<2:continue
        for i in range(0,max(1,len(ids)-ctx["seq_len"]),stride):
            s=ids[i:i+ctx["seq_len"]+1]
            if len(s)<2:continue
            s=s+[ctx["pad"]]*(ctx["seq_len"]+1-len(s))
            ex.append(s)
            if len(ex)>=max_examples:return ex
    return ex

def make_batch(data,ctx,batch_size,device):
    b=random.choices(data,k=batch_size) if len(data)<batch_size else random.sample(data,batch_size)
    z=torch.tensor(b,dtype=torch.long,device=device)
    x=z[:,:-1];y=z[:,1:]
    return x,y,x.eq(ctx["pad"])

def decode(ids,ctx):
    bad={ctx["pad"],ctx["eos"],ctx["unk"]}
    return " ".join([ctx["itos"][int(i)] for i in ids if int(i) not in bad])

ctx=build_context(texts,cfg)
examples=make_lm_examples(texts,ctx,max_examples=20000,stride=32)
random.shuffle(examples)
n=int(0.8*len(examples));v=int(0.1*len(examples))
train_data=examples[:n];valid_data=examples[n:n+v];test_data=examples[n+v:]
print("train:",len(train_data),"valid:",len(valid_data),"test:",len(test_data),"vocab:",len(ctx["vocab"]))

train: 16000 valid: 2000 test: 2000 vocab: 12000


## Quantization and sparsity hooks

In [16]:
def qste(x,bits,use):
    if not use:return x
    qmax=2**(bits-1)-1;s=x.abs().max().clamp(min=1e-8)/qmax
    y=(x/s).round().clamp(-qmax,qmax)*s
    return x+(y-x).detach()
def sparsify(w,ratio,use):
    if not use or ratio<=0:return w
    th=torch.quantile(w.abs().flatten(),ratio)
    return w*(w.abs()>=th)
def qw(w,cfg):return qste(sparsify(w,cfg["ws_ratio"],cfg["use_ws"]),cfg["w_bits"],cfg["use_wq"])
def qa(x,cfg):return qste(x,cfg["a_bits"],cfg["use_aq"])
def lin(x,l,cfg):return F.linear(qa(x,cfg),qw(l.weight,cfg),l.bias)

## Model

In [17]:
class MHA(nn.Module):
    def __init__(self,d_model,num_heads,cfg):
        super().__init__();self.h=num_heads;self.dh=d_model//num_heads;self.cfg=cfg
        self.q=nn.Linear(d_model,d_model);self.k=nn.Linear(d_model,d_model);self.v=nn.Linear(d_model,d_model);self.o=nn.Linear(d_model,d_model)
    def forward(self,x,key_pad=None):
        B,T,D=x.shape
        Q=lin(x,self.q,self.cfg).view(B,T,self.h,self.dh).transpose(1,2)
        K=lin(x,self.k,self.cfg).view(B,T,self.h,self.dh).transpose(1,2)
        V=lin(x,self.v,self.cfg).view(B,T,self.h,self.dh).transpose(1,2)
        S=Q@K.transpose(-2,-1)/math.sqrt(self.dh)
        S=S.masked_fill(torch.triu(torch.ones(T,T,device=x.device,dtype=torch.bool),1)[None,None,:,:],-1e9)
        if key_pad is not None:S=S.masked_fill(key_pad[:,None,None,:],-1e9)
        if self.cfg["attn_top_k"] is not None:
            k=min(self.cfg["attn_top_k"],T);th=S.topk(k,dim=-1).values[...,-1,None];S=S.masked_fill(S<th,-1e9)
        A=F.softmax(S,dim=-1);O=(A@V).transpose(1,2).contiguous().view(B,T,D)
        return lin(O,self.o,self.cfg),A

class FFN(nn.Module):
    def __init__(self,d_model,d_ff,cfg):
        super().__init__();self.l1=nn.Linear(d_model,d_ff);self.l2=nn.Linear(d_ff,d_model);self.cfg=cfg
    def forward(self,x):
        return lin(F.relu(lin(x,self.l1,self.cfg)),self.l2,self.cfg)

class Block(nn.Module):
    def __init__(self,d_model,d_ff,num_heads,cfg):
        super().__init__();self.a=MHA(d_model,num_heads,cfg);self.f=FFN(d_model,d_ff,cfg);self.n1=nn.LayerNorm(d_model);self.n2=nn.LayerNorm(d_model);self.cfg=cfg
    def forward(self,x,pad):
        if self.cfg["norm"]=="post":a,A=self.a(x,pad);x=self.n1(x+a);x=self.n2(x+self.f(x))
        else:n=self.n1(x);a,A=self.a(n,pad);x=x+a;x=x+self.f(self.n2(x))
        return x,A

class GPT2Small(nn.Module):
    def __init__(self,vocab_size,cfg,ctx):
        super().__init__();d=cfg["d_model"];self.cfg=cfg;self.ctx=ctx
        self.tok=nn.Embedding(vocab_size,d,padding_idx=ctx["pad"]);self.pos=nn.Embedding(ctx["seq_len"],d)
        self.blocks=nn.ModuleList([Block(d,cfg["d_ff"],cfg["num_heads"],cfg) for _ in range(cfg["num_layers"])])
        self.n=nn.LayerNorm(d);self.out=nn.Linear(d,vocab_size,bias=False)
    def forward(self,x,pad):
        p=torch.arange(x.size(1),device=x.device)[None,:];h=self.tok(x)+self.pos(p);A=[]
        for b in self.blocks:h,a=b(h,pad);A.append(a)
        return lin(self.n(h),self.out,self.cfg),A

## Train

In [18]:
model=GPT2Small(len(ctx["vocab"]),cfg,ctx).to(device)
opt=torch.optim.AdamW(model.parameters(),lr=cfg["lr"],weight_decay=cfg["weight_decay"])

def acc_ignore_pad_unk(logits,y,ctx):
    pred=logits.argmax(-1)
    m=y.ne(ctx["pad"])&y.ne(ctx["unk"])
    if m.sum().item()==0:return 0.0
    return (pred.eq(y)&m).sum().item()/m.sum().item()

def step(data,train=True):
    model.train(train)
    x,y,pad=make_batch(data,ctx,cfg["batch_size"],device)
    yloss=y.clone()
    yloss[yloss==ctx["unk"]]=ctx["pad"]
    with torch.set_grad_enabled(train):
        logits,A=model(x,pad)
        loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)),yloss.reshape(-1),ignore_index=ctx["pad"])
        if train:
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()
    return loss.item(),acc_ignore_pad_unk(logits,y,ctx)

def evaluate(data,n=20):
    loss=0;acc=0
    for _ in range(n):
        l,a=step(data,False);loss+=l;acc+=a
    return loss/n,acc/n

def train_epochs(epochs=200,train_steps=100,valid_steps=50,patience=15):
    history={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]}
    best=float("inf");best_state=None;bad=0
    for epoch in range(1,epochs+1):
        tl=ta=0
        for _ in range(train_steps):
            l,a=step(train_data,True)
            tl+=l;ta+=a
        vl,va=evaluate(valid_data,valid_steps)
        tl/=train_steps;ta/=train_steps
        history["train_loss"].append(tl);history["train_acc"].append(ta)
        history["val_loss"].append(vl);history["val_acc"].append(va)
        if vl<best:
            best=vl;bad=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            bad+=1
        print(epoch,"train_loss",tl,"train_acc",ta,"val_loss",vl,"val_acc",va)
        if bad>=patience:
            print("early stop at epoch",epoch,"best_val_loss",best)
            break
    return best_state,history

best_state,history=train_epochs(epochs=cfg["epochs"],train_steps=100,valid_steps=50,patience=15)
model.load_state_dict(best_state)

1 train_loss 7.757315063476563 train_acc 0.04347123655890763 val_loss 6.865595445632935 val_acc 0.056163200739541654
2 train_loss 6.803409457206726 train_acc 0.056166465152595865 val_loss 6.7504505443572995 val_acc 0.06371409459851676
3 train_loss 6.679728832244873 train_acc 0.07959362337946964 val_loss 6.557877407073975 val_acc 0.0961702678779093
4 train_loss 6.490031185150147 train_acc 0.09973170337375928 val_loss 6.4246416091918945 val_acc 0.10151992236016077
5 train_loss 6.298492488861084 train_acc 0.11183299154731019 val_loss 6.269437017440796 val_acc 0.11397307953860122
6 train_loss 6.211013722419739 train_acc 0.11801147274713752 val_loss 6.099850873947144 val_acc 0.12483936029238654
7 train_loss 6.0487536478042605 train_acc 0.12886778935435494 val_loss 5.999870357513427 val_acc 0.1301180314188212
8 train_loss 5.986421518325805 train_acc 0.13486010694774914 val_loss 5.927002544403076 val_acc 0.1345280264756462
9 train_loss 5.829294772148132 train_acc 0.14171832860161865 val_loss 

<All keys matched successfully>

## Generate

In [21]:
def generate(prompt,max_new=50):
    model.eval()
    ids=[ctx["stoi"].get(w,ctx["unk"]) for w in prompt.split()]
    for _ in range(max_new):
        x=ids[-ctx["seq_len"]:]
        x=x+[ctx["pad"]]*(ctx["seq_len"]-len(x))
        x=torch.tensor([x],dtype=torch.long,device=device)
        pad=x.eq(ctx["pad"])
        with torch.no_grad():
            logits,A=model(x,pad)
        pos=min(len(ids),ctx["seq_len"])-1
        logit=logits[0,pos].clone()
        logit[ctx["unk"]]=-1e9
        logit[ctx["pad"]]=-1e9
        probs=F.softmax(logit,dim=-1)
        nxt=int(probs.argmax())
        if nxt==ctx["eos"]:break
        ids.append(nxt)
    return decode(ids,ctx)

print(generate("Mary went to the",50))

went to the park. She saw a big box in the ground. It was a big tree and a big tree. The bird was a big and it was a big tree. The bird was very sad. She wanted to help. She asked her mom if she could do something to be a


## Later switches

In [20]:
# cfg["use_wq"]=True;cfg["use_aq"]=True
# cfg["use_ws"]=True;cfg["ws_ratio"]=0.5
# cfg["attn_top_k"]=8